# Day09：Test Agent

## Goal

在不修改真实 Unity 工程的前提下，将生成代码和 NUnit EditMode 测试注入临时沙箱，运行 Unity Test Framework 并得到结构化报告。

## Setup

默认使用 `CodingAgentTest` 和 Unity 2022.3。真实工程可以保持打开，因为 BatchMode 运行目标是临时副本。首次执行可能因导入 Package 耗时较长。

In [1]:
import json
import os
import sys
import tempfile
from pathlib import Path

workspace = Path.cwd().resolve()
if workspace.name == 'day09':
    workspace = workspace.parent
day06_path = workspace / 'day06'
if str(day06_path) not in sys.path:
    sys.path.insert(0, str(day06_path))

unity_path = Path(os.getenv(
    'UNITY_EDITOR_PATH',
    r'D:\Unity\Hub\Unity_Editor\2022.3.62f2c1\Editor\Unity.exe',
)).resolve()
unity_project_path = Path(os.getenv(
    'UNITY_TEST_PROJECT_PATH',
    r'D:\Unity\Unity_Project\CodingAgentTest',
)).resolve()
print('Unity:', unity_path)
print('真实工程:', unity_project_path)

Unity: D:\Unity\Hub\Unity_Editor\2022.3.62f2c1\Editor\Unity.exe
真实工程: D:\Unity\Unity_Project\CodingAgentTest


## Steps

### 1. 准备可预测的生产代码和测试

In [2]:
from tools.test_generation_tool import TestGenerationTool

probe_root = tempfile.TemporaryDirectory()
production_path = Path(probe_root.name) / 'generated'
test_source_path = Path(probe_root.name) / 'generated_tests'
production_path.mkdir()

(production_path / 'Day09Calculator.cs').write_text(
    '''namespace CodingAgent.Day09
{
    public static class Day09Calculator
    {
        public static int Add(int left, int right) => left + right;
        public static int Multiply(int left, int right) => left * right;
    }
}
''',
    encoding='utf-8',
)
test_write_result = TestGenerationTool(test_source_path).apply([
    {
        'name': 'Day09CalculatorTests.cs',
        'content': '''using NUnit.Framework;
using CodingAgent.Day09;

public class Day09CalculatorTests
{
    [Test]
    public void Add_ReturnsSum()
    {
        Assert.AreEqual(5, Day09Calculator.Add(2, 3));
    }

    [Test]
    public void Multiply_ReturnsProduct()
    {
        Assert.AreEqual(12, Day09Calculator.Multiply(3, 4));
    }
}
''',
    }
])
print(json.dumps(test_write_result, ensure_ascii=False, indent=2))

{
  "success": true,
  "files": [
    "Day09CalculatorTests.cs"
  ],
  "errors": []
}


### 2. 在隔离 Unity 工程中运行 EditMode 测试

In [3]:
from tools.unity_test_tool import UnityTestTool

real_tests_path = unity_project_path / 'Assets' / 'Tests'
real_tests_before = sorted(
    str(path.relative_to(unity_project_path))
    for path in real_tests_path.rglob('*')
) if real_tests_path.exists() else []

test_result = UnityTestTool(
    unity_path,
    unity_project_path,
    production_path,
    test_source_path,
    timeout=600,
).run()
print(json.dumps({
    'success': test_result['success'],
    'system_error': test_result['system_error'],
    'summary': test_result['summary'],
    'tests': test_result['tests'],
    'sandbox_cleaned': test_result['sandbox_cleaned'],
}, ensure_ascii=False, indent=2))

{
  "success": true,
  "system_error": false,
  "summary": {
    "total": 2,
    "passed": 2,
    "failed": 0,
    "skipped": 0,
    "inconclusive": 0,
    "duration": 0.0457118
  },
  "tests": [
    {
      "name": "Add_ReturnsSum",
      "full_name": "Day09CalculatorTests.Add_ReturnsSum",
      "result": "Passed",
      "duration": 0.013465,
      "message": "",
      "stack_trace": ""
    },
    {
      "name": "Multiply_ReturnsProduct",
      "full_name": "Day09CalculatorTests.Multiply_ReturnsProduct",
      "result": "Passed",
      "duration": 0.00025,
      "message": "",
      "stack_trace": ""
    }
  ],
  "sandbox_cleaned": true
}


## Checks

验证两个真实 NUnit 测试通过、报告可解析、沙箱已清理，并且真实工程的测试目录没有变化。

In [4]:
real_tests_after = sorted(
    str(path.relative_to(unity_project_path))
    for path in real_tests_path.rglob('*')
) if real_tests_path.exists() else []
assert test_write_result['success']
assert test_result['success']
assert not test_result['system_error']
assert test_result['summary']['total'] == 2
assert test_result['summary']['passed'] == 2
assert test_result['summary']['failed'] == 0
assert test_result['sandbox_cleaned']
assert real_tests_before == real_tests_after
probe_root.cleanup()
print('PASS: Day09 真实 Unity EditMode Test Agent 集成验收通过')

PASS: Day09 真实 Unity EditMode Test Agent 集成验收通过


## 独立验收：故障注入 → Repair → 重新测试

这一节使用全新的临时目录，不依赖上面的示例变量。先把 `Add` 的加法确定性地替换成减法，确认真实 Unity EditMode 测试失败；然后由 `RepairAgent` 调用工程化 `RepairTool` 生成并记录补丁，最后重新运行同一测试确认闭环通过。确定性 LLM 只负责返回固定修复代码，因此验收不需要联网，也不会受模型输出波动影响。

### 1. 注入故障并确认测试失败

In [5]:
from tools.test_generation_tool import TestGenerationTool
from tools.unity_test_tool import UnityTestTool

repair_probe_root = tempfile.TemporaryDirectory()
repair_production_path = Path(repair_probe_root.name) / 'generated'
repair_test_source_path = Path(repair_probe_root.name) / 'generated_tests'
repair_production_path.mkdir()

correct_calculator_source = '''namespace CodingAgent.Day09
{
    public static class Day09RepairCalculator
    {
        public static int Add(int left, int right) => left + right;
    }
}
'''
faulty_calculator_source = correct_calculator_source.replace(
    'left + right',
    'left - right',
)
repair_target_path = repair_production_path / 'Day09RepairCalculator.cs'
repair_target_path.write_text(faulty_calculator_source, encoding='utf-8')

repair_test_write_result = TestGenerationTool(repair_test_source_path).apply([
    {
        'name': 'Day09RepairCalculatorTests.cs',
        'content': '''using NUnit.Framework;
using CodingAgent.Day09;

public class Day09RepairCalculatorTests
{
    [Test]
    public void Add_ReturnsSum()
    {
        Assert.AreEqual(5, Day09RepairCalculator.Add(2, 3));
    }
}
''',
    }
])
repair_real_tests_before = sorted(
    str(path.relative_to(unity_project_path))
    for path in real_tests_path.rglob('*')
) if real_tests_path.exists() else []

first_repair_test_result = UnityTestTool(
    unity_path,
    unity_project_path,
    repair_production_path,
    repair_test_source_path,
    timeout=600,
).run()
print(json.dumps({
    'injected_change': 'left + right -> left - right',
    'success': first_repair_test_result['success'],
    'system_error': first_repair_test_result['system_error'],
    'summary': first_repair_test_result['summary'],
    'errors': first_repair_test_result['errors'],
    'sandbox_cleaned': first_repair_test_result['sandbox_cleaned'],
}, ensure_ascii=False, indent=2))

assert repair_test_write_result['success']
assert not first_repair_test_result['success']
assert not first_repair_test_result['system_error']
assert first_repair_test_result['summary']['total'] == 1
assert first_repair_test_result['summary']['failed'] == 1
assert first_repair_test_result['sandbox_cleaned']

{
  "injected_change": "left + right -> left - right",
  "success": false,
  "system_error": false,
  "summary": {
    "total": 1,
    "passed": 0,
    "failed": 1,
    "skipped": 0,
    "inconclusive": 0,
    "duration": 0.0472761
  },
  "errors": [
    {
      "test": "Day09RepairCalculatorTests.Add_ReturnsSum",
      "message": "  Expected: 5\n  But was:  -1\n",
      "stack_trace": "at Day09RepairCalculatorTests.Add_ReturnsSum () [0x00001] in C:\\Users\\admin\\AppData\\Local\\Temp\\coding-agent-unity-tests-tg5c_2is\\Project\\Assets\\Tests\\EditMode\\Day09RepairCalculatorTests.cs:9\n"
    }
  ],
  "sandbox_cleaned": true
}


### 2. 让 RepairAgent 通过 RepairTool 应用补丁

In [6]:
from agents.repair import RepairAgent
from memory.patch_history import PatchHistory
from tools.diff_tool import DiffTool
from tools.file_manager import FileManager
from tools.repair_tool import RepairTool

class DeterministicRepairLLM:
    def __init__(self, repaired_source):
        self.repaired_source = repaired_source
        self.last_prompt = ''

    def invoke(self, prompt):
        self.last_prompt = prompt
        return self.repaired_source

file_manager = FileManager()
repair_diff_tool = DiffTool(file_manager, repair_production_path)
repair_patch_history = PatchHistory(
    Path(repair_probe_root.name) / 'memory' / 'patch_history.json',
    repair_diff_tool,
)
repair_tool = RepairTool(
    file_manager,
    repair_production_path,
    repair_diff_tool,
    repair_patch_history,
)
repair_agent = RepairAgent(
    DeterministicRepairLLM(correct_calculator_source),
    repair_tool,
)
repair_result = repair_agent.run({
    'repair_count': 0,
    'root_causes': [
        {
            'type': 'test_failure',
            'target_file': 'Day09RepairCalculator.cs',
            'description': 'Add 应执行加法，但当前实现执行了减法。',
            'fix_action': {
                'operation': 'repair_test_failure',
                'target': 'Day09RepairCalculator.cs',
                'details': '将减法恢复为加法，同时保持公开 API 不变。',
            },
        }
    ],
    'review': {
        'remaining_issues': first_repair_test_result['errors'],
    },
})
repair_action = repair_result['repair_result']['actions'][0]
repair_records = repair_patch_history.list_records('Day09RepairCalculator.cs')
print(json.dumps({
    'repair_status': repair_result['repair_status'],
    'changed': repair_action['changed'],
    'patch_ids': repair_action['patch_ids'],
    'diff': repair_action['patches'][0]['diff'],
    'history_record_count': len(repair_records),
}, ensure_ascii=False, indent=2))

assert repair_result['repair_status'] == 'success'
assert repair_action['success']
assert repair_action['changed']
assert repair_action['patch_ids']
assert len(repair_records) == 1
assert 'left + right' in repair_target_path.read_text(encoding='utf-8')

[Repair Agent]开始执行
[Repair Agent]第1轮修复
[Repair Router]检测:repair_test_failure
[File Manager]写入完成:C:\Users\admin\AppData\Local\Temp\tmpbiz7nwg5\generated\Day09RepairCalculator.cs
{
  "repair_status": "success",
  "changed": true,
  "patch_ids": [
    "5ad7e4dc1cda4c0fa33fc451cc7a5421"
  ],
  "diff": "--- a/Day09RepairCalculator.cs\n+++ b/Day09RepairCalculator.cs\n@@ -2,6 +2,6 @@\n {\n     public static class Day09RepairCalculator\n     {\n-        public static int Add(int left, int right) => left - right;\n+        public static int Add(int left, int right) => left + right;\n     }\n }",
  "history_record_count": 1
}


### 3. 重新运行同一测试并验收闭环

In [7]:
second_repair_test_result = UnityTestTool(
    unity_path,
    unity_project_path,
    repair_production_path,
    repair_test_source_path,
    timeout=600,
).run()
repair_real_tests_after = sorted(
    str(path.relative_to(unity_project_path))
    for path in real_tests_path.rglob('*')
) if real_tests_path.exists() else []
print(json.dumps({
    'success': second_repair_test_result['success'],
    'system_error': second_repair_test_result['system_error'],
    'summary': second_repair_test_result['summary'],
    'sandbox_cleaned': second_repair_test_result['sandbox_cleaned'],
}, ensure_ascii=False, indent=2))

assert second_repair_test_result['success']
assert not second_repair_test_result['system_error']
assert second_repair_test_result['summary']['total'] == 1
assert second_repair_test_result['summary']['passed'] == 1
assert second_repair_test_result['summary']['failed'] == 0
assert second_repair_test_result['sandbox_cleaned']
assert repair_real_tests_before == repair_real_tests_after
repair_probe_root.cleanup()
print('PASS: Day09 故障注入 → Repair → 重新测试闭环验收通过')

{
  "success": true,
  "system_error": false,
  "summary": {
    "total": 1,
    "passed": 1,
    "failed": 0,
    "skipped": 0,
    "inconclusive": 0,
    "duration": 0.0416988
  },
  "sandbox_cleaned": true
}
PASS: Day09 故障注入 → Repair → 重新测试闭环验收通过


## Next Steps

- 完整工作流会让 Test Generator 根据本轮生产代码生成 EditMode 测试。
- Unity XML 报告进入 Reviewer；测试失败会进入修复循环，运行器错误则停止。
- Day10 将进入长期 Memory Agent；PlayMode 测试留作测试能力后续增强。